Forecasting Appointment Volume for Nexora Care Flow:

We can forcast the volume of clinic(s) appointment from the Nexora Care dataset. 

This projects to solves the volatile appointment demand of participating clinics thereby offering insights to administrators to have forward visibility into appointments, Staffing and room allocation. Because these clinics run tightly staffed, appointment-driven schedules rather than 24/7 shifts, even modest swings in daily demand have an outsized effect on wait times and staff workload. Administrators repeatedly report that they only learn a day is unusually busy once it is already underway

In [13]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print('Libraries imported successfully')

Libraries imported successfully


In [ ]:
# Load the appointment dataset
path = r'C:\Users\Joshua\amdira-projects\Appointment-volume-forecasting\data\AppointmentRecords.csv'
df = pd.read_csv(path)
print('Dataset loaded successfully')
print(f'Dataset has {df.shape[0]} rows and {df.shape[1]} columns')

print('Preview of the dataset:')
print(df.head().to_string())

Dataset loaded successfully
Dataset has 129353 rows and 15 columns
Preview of the dataset:
   AppointmentID  ClinicID                      ClinicName          ClinicType  PatientID  ProviderID AppointmentDateTime AppointmentType     Status BookingChannel  BookingLeadTimeDays  IsHoliday     HolidayName  SeasonFlag  ChronicConditionFlag
0         529385         2          Nexora Care - Lakeside     Multi-Specialty     103642         202    2024-01-01 08:00     New Patient  Completed         Online                    2       True  New Year's Day  Flu Season                 False
1         500000         1         Nexora Care - Riverside        Primary Care     101551         101    2024-01-01 08:30     New Patient    No-Show         Online                   18       True  New Year's Day  Flu Season                 False
2         500011         1         Nexora Care - Riverside        Primary Care     100260         104    2024-01-01 08:30       Follow-Up  Completed       Referral        

In [21]:
print('Dataset information:')
print(df.info())

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129353 entries, 0 to 129352
Data columns (total 15 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   AppointmentID         129353 non-null  int64 
 1   ClinicID              129353 non-null  int64 
 2   ClinicName            129353 non-null  object
 3   ClinicType            129353 non-null  object
 4   PatientID             129353 non-null  int64 
 5   ProviderID            129353 non-null  int64 
 6   AppointmentDateTime   129353 non-null  object
 7   AppointmentType       129353 non-null  object
 8   Status                129353 non-null  object
 9   BookingChannel        129353 non-null  object
 10  BookingLeadTimeDays   129353 non-null  int64 
 11  IsHoliday             129353 non-null  bool  
 12  HolidayName           939 non-null     object
 13  SeasonFlag            129353 non-null  object
 14  ChronicConditionFlag  129353 non-null  bool  
d

In [30]:
print('Inspecting the dataset... ')

print('Percentage of null values in each column:')
print(df.isnull().sum()/df.shape[0]*100)
print('\nChecking for duplicate rows in the dataset...')
print(f'Number of duplicate rows: {df.duplicated().sum()}')
print('Checking for duplicated AppointmentIDs...')
print(f'Number of duplicated AppointmentIDs: {df["AppointmentID"].duplicated().sum()}')

Inspecting the dataset... 
Percentage of null values in each column:
AppointmentID            0.000000
ClinicID                 0.000000
ClinicName               0.000000
ClinicType               0.000000
PatientID                0.000000
ProviderID               0.000000
AppointmentDateTime      0.000000
AppointmentType          0.000000
Status                   0.000000
BookingChannel           0.000000
BookingLeadTimeDays      0.000000
IsHoliday                0.000000
HolidayName             99.274079
SeasonFlag               0.000000
ChronicConditionFlag     0.000000
dtype: float64

Checking for duplicate rows in the dataset...
Number of duplicate rows: 0
Checking for duplicated AppointmentIDs...
Number of duplicated AppointmentIDs: 0


In [35]:
# Validating Dates and time columns
print('Checking for valid date formats in the dataset...')
df['AppointmentDateTime'] = pd.to_datetime(df['AppointmentDateTime'], errors='coerce')
print('Invalid dates in AppointmentDateTime column:')
print(df[df['AppointmentDateTime'].isnull()].shape[0])

Checking for valid date formats in the dataset...
Invalid dates in AppointmentDateTime column:
0


In [49]:
# Checking for Categorical variables and their unique values
 
categorical_columns = [
    "ClinicName",
    "ClinicType",
    "AppointmentType",
    "Status",
    "BookingChannel",
    "SeasonFlag"
]
for col in categorical_columns:
    print(f'Column: {col}')
    print(f'Unique values: {df[col].unique()}')
    print(f'Number of unique values: {df[col].nunique()}')
    print(f'Value counts:\n{df[col].value_counts(dropna=False)}')
    print('---')



Column: ClinicName
Unique values: ['Nexora Care - Lakeside' 'Nexora Care - Riverside'
 'Nexora Care - Downtown Express' 'Nexora Care - Hillcrest']
Number of unique values: 4
Value counts:
Nexora Care - Lakeside            45681
Nexora Care - Riverside           29382
Nexora Care - Hillcrest           29144
Nexora Care - Downtown Express    25146
Name: ClinicName, dtype: int64
---
Column: ClinicType
Unique values: ['Multi-Specialty' 'Primary Care' 'Telehealth-Forward']
Number of unique values: 3
Value counts:
Primary Care          58526
Multi-Specialty       45681
Telehealth-Forward    25146
Name: ClinicType, dtype: int64
---
Column: AppointmentType
Unique values: ['New Patient' 'Follow-Up' 'Urgent' 'Telehealth']
Number of unique values: 4
Value counts:
Follow-Up      57859
Telehealth     30117
New Patient    21583
Urgent         19794
Name: AppointmentType, dtype: int64
---
Column: Status
Unique values: ['Completed' 'No-Show' 'Cancelled' 'Rescheduled']
Number of unique values: 4
Value 

In [50]:
    # Checking for leading/trailing spaces in categorical variables
    print(f'Checking for leading/trailing spaces in column: {col}')
    if df[col].dtype == 'object':
        if df[col].str.contains('^\s+|\s+$', regex=True).any():
            print(f'Column {col} contains leading/trailing spaces in some values.')
        else:
            print(f'Column {col} does not contain leading/trailing spaces in any values.')

Checking for leading/trailing spaces in column: SeasonFlag
Column SeasonFlag does not contain leading/trailing spaces in any values.


In [25]:
print('Summary statistics of the dataset:')
print(df.describe().to_string())

Summary statistics of the dataset:
       AppointmentID       ClinicID      PatientID     ProviderID  BookingLeadTimeDays
count  129353.000000  129353.000000  129353.000000  129353.000000        129353.000000
mean   564676.000000       2.417864  103553.485385     245.290005             8.704267
std     37341.139023       1.071525    2020.941152     107.165883             7.641921
min    500000.000000       1.000000  100000.000000     101.000000             0.000000
25%    532338.000000       2.000000  101812.000000     201.000000             3.000000
50%    564676.000000       2.000000  103547.000000     205.000000             7.000000
75%    597014.000000       3.000000  105178.000000     306.000000            12.000000
max    629352.000000       4.000000  107279.000000     406.000000            60.000000
